In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "CS340ISFUN"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# drop the _id column 
df.drop(columns=['_id'],inplace=True)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash('SimpleExample')


image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#buttons for filtering potential search and rescue dogs. 
app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Div(html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'width': '200px'}),
                    href='https://www.snhu.edu', target='_blank'),
                    style ={'display': 'flex', 'justifyContent': 'center'}),
             
    html.Center(html.B(html.H1("Travis Erwin's CS-340 Dashboard"))),
   
                                                                                   
    html.Hr(),
    html.Div([
        dcc.RadioItems(
            id='rescue-type-filter',
            options=[
                {'label':'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster/Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}],
            value = 'Reset',
            inline = True
        )]
    ),
      
#Datatable
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True, "filter_options":{'case':'insensitive'}} for i in df.columns #added filter_options to eliminate case sensitivity when filtering. 
        ],
        data=df.to_dict('records'),
        
        row_selectable="single",
        selected_rows=[0],
        #pagination
        page_action="native",
        page_current=0,
        page_size=10,  #limits results to 10 per page
        #row sorting
        sort_action="native",
        sort_mode="multi", #added for sorting by multiple categories at once.
        filter_action="native" #filters results by keyword in a column. Filtering caused index errors, had to solve for that in the callback.

                        ),
    html.Br(),
    html.Hr(),
    
#Setup map and pie chart side by side as well as center them for a cleaner look.
    html.Div(className='row',
         style={'display' : 'flex', 'justifyContent': 'center', 'alignItems': 'flex-start','width':'100%'},
             children=[
        html.Div(
            id='graph-id',
            style={'display': 'flex', 'justifyContent' : 'center', 'width': '50%'},
            

            ),
        html.Div(
            id='map-id',
            style={'display': 'flex', 'justifyContent' : 'center', 'width': '50%'}
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Callback for filtering options when user clicks on radio button. 
@app.callback(Output('datatable-id','data'),
              [Input('rescue-type-filter', 'value')])

#filters dogs suitable for water rescue
def update_dashboard(filter_type):
    if filter_type == "Water": query = {
        'animal_type': "Dog", 'breed':{ '$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
        'sex_upon_outcome': 'Intact Female',
        'age_upon_outcome_in_weeks': {'$gte':26, '$lte':156}
    }
    
#fitlers dogs suitable for mountain and wilderness rescue
    elif filter_type == 'Mountain': query = {
        'animal_type': "Dog", 'breed':{ '$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']},
        'sex_upon_outcome': 'Intact Male',
        'age_upon_outcome_in_weeks': {'$gte':26, '$lte':156}
    }
    
#filters dogs suitable for disaster and indiviual rescue
    elif filter_type == 'Disaster': query = {
        'animal_type': "Dog", 'breed':{ '$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler']},
        'sex_upon_outcome': 'Intact Male',
        'age_upon_outcome_in_weeks': {'$gte':20, '$lte':300}
    }

#clears radio button filters. 
    else:
        query = {}
        
    df = pd.DataFrame.from_records(db.read(query))
    df.drop(columns=['_id'], inplace=True, errors='ignore')
    return df.to_dict('records')
        
# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
    Input('rescue-type-filter', 'value')])
def update_graphs(viewData, filter_type):
    
    if viewData is None:
        return[]
    
    df = pd.DataFrame.from_dict(viewData)
    
    if df.empty:
        return[]
    
# Limit the pie chart to top 10 returned breeds to limit clutter. 
    top_breeds = df['breed'].value_counts().nlargest(10).index
    df = df[df['breed'].isin(top_breeds)]

# Name the pie chart based on the rescue filter selected.
    if filter_type == 'Water':
        chart_title = 'Dogs Suitable For Water Rescue'
    elif filter_type == 'Mountain':
        chart_title = 'Dogs Suitable for Mountain and Wilderness Rescue'
    elif filter_type == 'Disaster':
        chart_title = 'Dogs Suitable for Disaster or Individual Tracking'
    else:
        chart_title = 'Shelter Animals (limited to top ten)'

# Return the pie chart
    return [
        dcc.Graph(            
            figure = px.pie(df, names='breed', title= chart_title)
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
     #added if statement to handle the nonetype error I was recieving when starting up my dashboard.
    if selected_columns is None:
        return[]
    
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    dff = pd.DataFrame.from_dict(viewData)
    
    #if datatable is empty after filtering, "no data" is printed. 
    if dff.empty:
        return[html.P("No Data")]
    
    #uses first row if nothing is selected, otherwise ensure the selected index is within the index range, this solves the index error I was recieving when filtering. 
    row = 0 if not index else min(index[0], len(dff)-1)
        
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[dff.iloc[row,13],dff.iloc[row,14]], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal, hover over the marker and it shows the animal breed
            # Column 9 defines the name of the animal, click the marker and it shows the animal breed and name
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://othellogalileo-idiomaxiom-3000.codio.io/proxy/8050/
